# 08 — Model Testing / Prediction
## Paddy Yield Predictor

Load the saved model and test it on a real row from the dataset.
This confirms the saved pipeline works end-to-end before we use it in the app.

> **Note:** The original notebook had a bug — it used undefined variable names (`pipeline` and `predicted`). Both are fixed here.

In [6]:
import sys
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.logger import get_logger
from src.model_utils import load_model

log = get_logger('08_model_testing')
log.info('Starting Model Testing notebook')

2026-08-18 15:21:49 | INFO     | 08_model_testing | Starting Model Testing notebook


In [7]:
# Load the saved model
try:
    model_path = PROJECT_ROOT / 'models' / 'paddy_yield_predictor.pkl'
    model = load_model(model_path)
    print('Model loaded successfully.')
except FileNotFoundError:
    log.error('Model file not found. Run notebook 06 first to train and save the model.')
    raise
except Exception as e:
    log.error(f'Could not load model: {e}')
    raise

2026-08-18 15:21:49 | INFO     | src.model_utils | Model loaded from: d:\TRUPTIMAYEE KHUNTIA\PADDY YIELD PREDICTOR\paddy_project\models\paddy_yield_predictor.pkl


Model loaded successfully.


In [8]:
# Load the dataset to pick a real sample row
try:
    df = pd.read_csv(PROJECT_ROOT / 'paddydataset.csv')
    df.columns = df.columns.astype(str).str.strip()

    TARGET = 'Paddy yield(in Kg)'
    X = df.drop(columns=[TARGET])
    y = df[TARGET]

    print('Dataset loaded. Using row 0 as a test sample.')
except Exception as e:
    log.error(f'Failed to load dataset: {e}')
    raise

Dataset loaded. Using row 0 as a test sample.


In [9]:
# Pick the first row as our sample to predict
try:
    sample = X.iloc[[0]]          # shape: (1, 44)
    actual_yield = y.iloc[0]      # the real answer

    predicted_yield = float(model.predict(sample)[0])

    print(f'Actual yield    : {actual_yield:,.2f} Kg')
    print(f'Predicted yield : {predicted_yield:,.2f} Kg')
    print(f'Difference      : {abs(actual_yield - predicted_yield):,.2f} Kg')

    log.info(f'Sample prediction — Actual: {actual_yield:.2f} | Predicted: {predicted_yield:.2f}')

except Exception as e:
    log.error(f'Prediction failed: {e}')
    raise

2026-08-18 15:21:49 | INFO     | 08_model_testing | Sample prediction — Actual: 35028.00 | Predicted: 35609.08


Actual yield    : 35,028.00 Kg
Predicted yield : 35,609.08 Kg
Difference      : 581.08 Kg


In [10]:
# Run predictions on a few more rows to spot-check
try:
    sample_5 = X.iloc[:5]
    actual_5 = y.iloc[:5].values
    predicted_5 = model.predict(sample_5)

    check = pd.DataFrame({
        'Actual (Kg)'   : actual_5.round(2),
        'Predicted (Kg)': predicted_5.round(2),
        'Diff (Kg)'     : abs(actual_5 - predicted_5).round(2)
    })
    display(check)
    log.info('Spot-check on 5 samples completed')

except Exception as e:
    log.error(f'Spot-check failed: {e}')
    raise

,Actual (Kg),Predicted (Kg),Diff (Kg)
0,35028,35609.08,581.08
1,35412,35834.79,422.79
2,36300,36230.36,69.64
3,35016,35146.61,130.61
4,34044,35816.09,1772.09


2026-08-18 15:21:50 | INFO     | 08_model_testing | Spot-check on 5 samples completed


### ✅ Pipeline works
The model loaded correctly and predictions look reasonable.
You can now run `streamlit run app.py` to use the model through a web interface.